# Heart Disease Prediction 3 Machine Learning Models

Training and comparing 3 models:
1. XGBoost
2. Random Forest 
3. (TBD)

## 1. Import Libraries

In [1]:
# ---- IMPORTS ----

# pandas = library for reading CSV files and working with tables (called DataFrames)
# We use it to load main_dataset.csv and manipulate rows/columns
# "as pd" = shortcut so we type pd instead of pandas
import pandas as pd

# numpy = library for working with numbers, arrays, and math operations
# We use it for numerical calculations
# "as np" = shortcut so we type np instead of numpy
import numpy as np

# train_test_split = function that splits our data into two groups:
#   - Training set (80%) = model learns from this
#   - Testing set (20%) = model is tested on this (never seen before)
# Why split? If we test on same data we trained on, model just memorizes answers
from sklearn.model_selection import train_test_split

# accuracy_score = calculates what percentage of predictions were correct
#   Example: 161 correct out of 184 = 0.875 = 87.5%
# classification_report = detailed report with precision, recall, f1-score
#   precision = of all predicted "disease", how many actually have it?
#   recall = of all actual "disease" patients, how many did we catch?
#   f1-score = balance between precision and recall
# confusion_matrix = 2x2 table showing correct vs wrong predictions
#   True Positive, True Negative, False Positive, False Negative
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# XGBClassifier = XGBoost model (eXtreme Gradient Boosting)
# How it works: builds 100 decision trees ONE AFTER ANOTHER
# Each new tree focuses on fixing mistakes of previous trees
# Final prediction = combined vote of all 100 trees
from xgboost import XGBClassifier

# RandomForestClassifier = Random Forest model
# How it works: builds 100 decision trees INDEPENDENTLY (at same time)
# Each tree sees a random subset of data and features
# Final prediction = majority vote of all trees
from sklearn.ensemble import RandomForestClassifier

# import TBD?


## 2. Load and Split Data

In [2]:
# pd.read_csv() = read CSV file and create a DataFrame (table)
# DataFrame = table with rows (patients) and columns (features)
# Our dataset: 918 rows (patients), 9 columns (8 features + 1 target)
df = pd.read_csv('main_dataset.csv')

# len(df) = count number of rows in the DataFrame
# len(df.columns) = count number of columns
# f"    " = f-string, allows putting variables inside text with {}
print(f"Dataset: {len(df)} rows, {len(df.columns)} columns")

# X = features (the 8 measurements we know about each patient)
# drop('target', axis=1) = remove the 'target' column from the table
#   axis=1 means drop a COLUMN (axis=0 would drop a ROW)
# Result: X has 8 columns: cp, thalach, oldpeak, exang, chol, slope, age, sex
# These are the INPUTS to our model (what we feed in)
X = df.drop('target', axis=1)

# y = target (what we want to predict)
# df['target'] = get just the 'target' column
# Values: 0 = no heart disease, 1 = has heart disease
# This is the OUTPUT we want our model to predict
y = df['target']

# train_test_split() splits data into 4 parts:
#   X_train = features for training (734 rows) - model LEARNS from this
#   X_test  = features for testing (184 rows) - model is TESTED on this
#   y_train = correct answers for training patients
#   y_test  = correct answers for testing patients (to check if model is right)
# test_size=0.2 = 20% goes to testing, 80% goes to training
#   918 * 0.2 = ~184 test rows, 918 * 0.8 = ~734 train rows
# random_state=333 = seed for random split, ensures same split every time
#   Different number = different split, but consistent across runs
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=333)

# Prints summary of what we have
# list(X.columns) = convert column names to a list for display
print(f"Features: {list(X.columns)}")
print(f"Training: {len(X_train)} rows")
print(f"Testing: {len(X_test)} rows")
# value_counts() = count how many 0s and 1s in target column
# to_dict() = convert to dictionary: {1: 517, 0: 401}
# Shows: 517 patients have disease, 401 do not 
print(f"Target: {y.value_counts().to_dict()}")

Dataset: 918 rows, 9 columns
Features: ['cp', 'thalach', 'oldpeak', 'exang', 'chol', 'slope', 'age', 'sex']
Training: 734 rows
Testing: 184 rows
Target: {1: 517, 0: 401}


---
## 3. Model 1: XGBoost

XGBoost = eXtreme Gradient Boosting
- Builds 100 decision trees one after another
- Each tree fixes mistakes of previous trees
- All trees vote together for final prediction

In [3]:
# Create XGBoost model
xgb_model = XGBClassifier(
    n_estimators=100,       # 100 decision trees
    max_depth=5,            # each tree max 5 levels deep
    learning_rate=0.1,      # small corrections each step
    random_state=42,        # reproducible results
    eval_metric='logloss'   # error measurement method
)

# Train - model learns patterns from 734 patients
print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# Predict - model predicts 184 test patients
xgb_pred = xgb_model.predict(X_test)

# Accuracy - how many predictions were correct
xgb_accuracy = accuracy_score(y_test, xgb_pred)
print(f"\nXGBoost Accuracy: {xgb_accuracy*100:.1f}%")

Training XGBoost...

XGBoost Accuracy: 83.7%


In [4]:
# XGBoost - Confusion Matrix
# Shows: correct predictions vs wrong predictions
cm = confusion_matrix(y_test, xgb_pred)
print("Confusion Matrix:")
print(f"                  Predicted")
print(f"                  No Disease  |  Has Disease")
print(f"  Actual No Disease:   {cm[0][0]}      |      {cm[0][1]}")
print(f"  Actual Has Disease:  {cm[1][0]}      |      {cm[1][1]}")

# Detailed report: precision, recall, f1-score
print(f"\nDetailed Report:")
print(classification_report(y_test, xgb_pred, target_names=['No Disease', 'Has Disease']))

Confusion Matrix:
                  Predicted
                  No Disease  |  Has Disease
  Actual No Disease:   64      |      14
  Actual Has Disease:  16      |      90

Detailed Report:
              precision    recall  f1-score   support

  No Disease       0.80      0.82      0.81        78
 Has Disease       0.87      0.85      0.86       106

    accuracy                           0.84       184
   macro avg       0.83      0.83      0.83       184
weighted avg       0.84      0.84      0.84       184



In [5]:
# XGBoost - Feature Importance
# Which features does the model use most?
print("Feature Importance:")
importance = xgb_model.feature_importances_
feature_importance = sorted(zip(X.columns, importance), key=lambda x: x[1], reverse=True)
for i, (feature, score) in enumerate(feature_importance, 1):
    bar = "█" * int(score * 50)
    print(f"  {i}. {feature:10s} {score:.4f} {bar}")

Feature Importance:
  1. slope      0.6211 ███████████████████████████████
  2. cp         0.1261 ██████
  3. exang      0.0617 ███
  4. sex        0.0595 ██
  5. oldpeak    0.0434 ██
  6. thalach    0.0360 █
  7. chol       0.0278 █
  8. age        0.0243 █


---
## 4. Model 2: Random Forest 

Random Forest:
-

In [6]:
# TODO:

---
## 5. Model 3:  (TBD)



In [7]:
# TODO: #


---
## 6. Compare All 3 Models

In [8]:
print("MODEL COMPARISON")
print("=" * 50)
print(f"\n  1. XGBoost:             {xgb_accuracy*100:.1f}%")
print(f"  2. Random Forest:       TBD")
print(f"  3. Logistic Regression: TBD")
print(f"\n" + "=" * 50)

MODEL COMPARISON

  1. XGBoost:             83.7%
  2. Random Forest:       TBD
  3. Logistic Regression: TBD





**Completed:**
- ✅ XGBoost has ~87.5% accuracy

**To Do:**
- ⬜ Random Forest
- ⬜ 1 more model (TBD)
- ⬜ Final comparison